# Final Evaluation: Warmup Learning Curves

Evaluate trained DQN agents against a heuristic baseline on parameterized
`ScalableEvalScenario` scenarios with **physics-guaranteed feasibility**.

**Pipeline:**
1. Load pre-trained agents from tutorial checkpoints
2. (Optional) Fine-tune on urgency scenarios (S32/S33)
3. Run warmup training on each scenario, recording **per-epoch learning curves**
4. Run heuristic baseline once per scenario
5. Plot agent adaptation curves vs. heuristic baseline
6. Focus on **sub-task completion percentages** and trends

In [1]:
# ================================================================
# Cell 1: Imports & Configuration
# ================================================================
import os, sys, random, time, warnings
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

# Add project root to path (notebook is in notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
})
warnings.filterwarnings('ignore', category=FutureWarning)

# Project imports
from simulation.training.curriculum_trainer import (
    create_env_factory, build_agent_config,
)
from simulation.rl.agent_registry import create_agent
from simulation.training.tutorial_runner import TutorialRunner
from simulation.training.scenarios import SCENARIO_BY_ID
from simulation.training.eval import (
    ScalableEvalScenario, EvalParams, HeuristicAgent,
)

# ── User configuration ────────────────────────────────────────
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Agents to evaluate: list of (dqn_variant, cnn_backbone) tuples.
AGENTS_TO_EVAL = [
    ("baseline",      "baseline"),
    # ("spectral_norm", "wider"),
    # ("munchausen",    "baseline"),
    # ("noisynet",      "region_aware"),
]

def agent_label(variant: str, backbone: str) -> str:
    return f"{variant}_{backbone}"

# Paths
CKPT_DIR = PROJECT_ROOT / 'runs' / 'tutorial_checkpoints'
OUTPUT_DIR = PROJECT_ROOT / 'runs' / 'final_eval'
FIG_DIR = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Warmup settings (per-combo adaptation — this IS the evaluation)
WARMUP_EPOCHS = 30         # Training episodes per scenario combo
WARMUP_EPSILON = 0.10      # Exploration during warmup
WARMUP_GRAD_STEPS = 4      # Gradient steps per warmup episode

# Training from scratch (fallback if no checkpoint)
TRAIN_EPOCHS = 800
MASTERY_THRESHOLD = 0.9

# Fine-tuning (optional, Cell 4)
FINETUNE_EPOCHS = 50
FINETUNE_EPSILON = 0.05
FINETUNE_SCENARIOS = [32, 33]

# Eval grid: imports up to 100, exports up to 100 → total up to 200
LOAD_SIZES = [20, 40, 80, 120, 200]
YARD_FILLS = [0.0, 0.25, 0.50]
BASE_SEED = 7000

print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')
print(f'Agents ({len(AGENTS_TO_EVAL)} combos):')
for v, b in AGENTS_TO_EVAL:
    print(f'  {agent_label(v, b):30s}  (DQN={v}, CNN={b})')
print(f'Warmup: {WARMUP_EPOCHS} epochs @ ε={WARMUP_EPSILON}, '
      f'{WARMUP_GRAD_STEPS} grad steps/epoch')
print(f'Grid: {len(LOAD_SIZES)} sizes × {len(YARD_FILLS)} fills '
      f'= {len(LOAD_SIZES) * len(YARD_FILLS)} combos')
print(f'Max containers: {max(LOAD_SIZES)} '
      f'(imp={max(LOAD_SIZES)//2} + exp={max(LOAD_SIZES)//2})')

Project root: /home/franko/CT-DRL-MA
Device: cuda
Agents (1 combos):
  baseline_baseline               (DQN=baseline, CNN=baseline)
Warmup: 30 epochs @ ε=0.1, 4 grad steps/epoch
Grid: 5 sizes × 3 fills = 15 combos
Max containers: 200 (imp=100 + exp=100)


In [2]:
# ================================================================
# Cell 2: Utilities
# ================================================================

def seed_everything(seed=SEED):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def load_agent_from_checkpoint(agent, ckpt_path: str):
    """Load checkpoint into agent."""
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    agent.q_net.load_state_dict(ckpt['q_net'])
    agent.target_net.load_state_dict(ckpt['target_net'])
    agent.optimizer.load_state_dict(ckpt['optimizer'])
    agent.step_count = ckpt['step_count']
    print(f'  Loaded checkpoint (step {agent.step_count})')


# Environment and config factories
ROWS, BAYS, TIERS = 5, 58, 5
TRACKS, SPLIT_FACTOR = 7, 20

env_factory = create_env_factory(
    rows=ROWS, bays=BAYS, tiers=TIERS,
    tracks=TRACKS, split_factor=SPLIT_FACTOR,
    export_ratio=1.0, max_retries=2,
)

cfg = build_agent_config(
    rows=ROWS, bays=BAYS, tiers=TIERS,
    split_factor=SPLIT_FACTOR, tracks=TRACKS,
)

seed_everything(SEED)
print('Utilities ready.')

Utilities ready.


In [3]:
# ================================================================
# Cell 3: Agent Loading / Training
# ================================================================

agents: Dict[str, object] = {}

for variant, backbone in AGENTS_TO_EVAL:
    label = agent_label(variant, backbone)
    print(f'\n{"="*60}')
    print(f'Agent: {label} (DQN={variant}, CNN={backbone})')
    print(f'{"="*60}')

    agent = create_agent(variant, cfg, backbone_variant=backbone)

    # Try loading a checkpoint (keyed by combo label)
    ckpt_path = CKPT_DIR / f'{label}_best.pt'
    alt_path = CKPT_DIR / f'{label}.pt'

    loaded = False
    for path in [ckpt_path, alt_path]:
        if path.exists():
            print(f'Loading checkpoint: {path}')
            load_agent_from_checkpoint(agent, str(path))
            loaded = True
            break

    if not loaded:
        print(f'No checkpoint found. Training from scratch ({TRAIN_EPOCHS} epochs)...')
        runner = TutorialRunner(
            env_factory=env_factory,
            agent_or_config=agent,
            verbose=True,
        )
        runner.train_all(epochs=TRAIN_EPOCHS, mastery_threshold=MASTERY_THRESHOLD)

        # Save checkpoint
        CKPT_DIR.mkdir(parents=True, exist_ok=True)
        agent.save(str(ckpt_path))
        print(f'Saved checkpoint: {ckpt_path}')

    agents[label] = agent
    print(f'{label}: ready (step_count={agent.step_count})')

print(f'\nLoaded {len(agents)} DQN agent(s).')


Agent: baseline_baseline (DQN=baseline, CNN=baseline)
Loading checkpoint: /home/franko/CT-DRL-MA/runs/tutorial_checkpoints/baseline_baseline_best.pt
  Loaded checkpoint (step 42209)
baseline_baseline: ready (step_count=42209)

Loaded 1 DQN agent(s).


In [4]:
# ================================================================
# Cell 4: Fine-tuning on S32/S33 (Optional — skip if not needed)
# ================================================================

SKIP_FINETUNE = True  # Set to False to run fine-tuning

if not SKIP_FINETUNE:
    for label, agent in agents.items():
        if not getattr(agent, 'is_trainable', True):
            continue

        print(f'\nFine-tuning {label} on S{FINETUNE_SCENARIOS}...')
        agent.epsilon_override = FINETUNE_EPSILON
        if hasattr(agent, 'set_noise_scale'):
            agent.set_noise_scale(0.1)

        runner = TutorialRunner(
            env_factory=env_factory,
            agent_or_config=agent,
            verbose=False,
        )

        for epoch in range(FINETUNE_EPOCHS):
            for sid in FINETUNE_SCENARIOS:
                sc = SCENARIO_BY_ID[sid]
                result = runner.run_scenario(sc)
                if hasattr(agent, 'replay') and hasattr(agent.replay, 'is_ready'):
                    if agent.replay.is_ready(agent.cfg.training.batch_size):
                        for _ in range(4):
                            agent.optimize()

            if (epoch + 1) % 10 == 0:
                old_eps = agent.epsilon_override
                agent.epsilon_override = 0.0
                val_results = []
                for sid in FINETUNE_SCENARIOS:
                    sc = SCENARIO_BY_ID[sid]
                    r = runner.run_scenario(sc)
                    val_results.append(('PASS' if r.passed else 'FAIL', r.total_reward))
                agent.epsilon_override = old_eps
                val_str = '  '.join(
                    f'S{sid}={tag} R={rew:+.1f}'
                    for sid, (tag, rew) in zip(FINETUNE_SCENARIOS, val_results)
                )
                print(f'  Epoch {epoch+1:3d}/{FINETUNE_EPOCHS}: {val_str}')

        agent.epsilon_override = None
        if hasattr(agent, 'set_noise_scale'):
            agent.set_noise_scale(1.0)

    print('\nFine-tuning complete.')
else:
    print('Fine-tuning skipped (SKIP_FINETUNE=True).')

Fine-tuning skipped (SKIP_FINETUNE=True).


In [5]:
# ================================================================
# Cell 5: Heuristic Baseline & Eval Grid
# ================================================================

heuristic = HeuristicAgent(cfg.unified)
print('Heuristic baseline created.')

# Build evaluation grid
def build_eval_grid(
    load_sizes=LOAD_SIZES,
    yard_fills=YARD_FILLS,
    base_seed=BASE_SEED,
) -> List[EvalParams]:
    """Build eval grid: load sizes x yard fill levels (train-only)."""
    grid = []
    for i, n in enumerate(load_sizes):
        for j, fill in enumerate(yard_fills):
            n_imp = n // 2
            n_exp = n - n_imp
            grid.append(EvalParams(
                n_imports=n_imp,
                n_exports=n_exp,
                yard_fill_pct=fill,
                seed=base_seed + i * 100 + j * 10,
            ))
    return grid

eval_grid = build_eval_grid()

print(f'\nEvaluation grid: {len(eval_grid)} parameter combos')
print(f'{"Idx":>4s}  {"Label":40s}  {"Total":>5s}  {"Trains":>6s}  '
      f'{"Window":>8s}  {"MaxSteps":>8s}')
print('-' * 85)
for i, p in enumerate(eval_grid):
    sc = ScalableEvalScenario(p)
    print(
        f'  [{i+1:2d}]  {p.label:40s}  {p.total_containers:5d}  '
        f'{sc._n_trains:6d}  '
        f'{sc._feasible_seconds/60:7.0f}min  '
        f'{sc.max_steps:8d}'
    )

Heuristic baseline created.

Evaluation grid: 15 parameter combos
 Idx  Label                                     Total  Trains    Window  MaxSteps
-------------------------------------------------------------------------------------
  [ 1]  imp=10_exp=10_fill=0%                        20       1       89min        69
  [ 2]  imp=10_exp=10_fill=25%                       20       1       96min        75
  [ 3]  imp=10_exp=10_fill=50%                       20       1       99min        78
  [ 4]  imp=20_exp=20_fill=0%                        40       1      199min       138
  [ 5]  imp=20_exp=20_fill=25%                       40       1      216min       153
  [ 6]  imp=20_exp=20_fill=50%                       40       1      219min       156
  [ 7]  imp=40_exp=40_fill=0%                        80       2      397min       276
  [ 8]  imp=40_exp=40_fill=25%                       80       2      432min       306
  [ 9]  imp=40_exp=40_fill=50%                       80       2      439min   

In [6]:
# ================================================================
# Cell 6: Warmup Learning Curves Collection
# ================================================================

def run_warmup_curves(
    agent,
    env_factory: Callable,
    param_grid: List[EvalParams],
    warmup_epochs: int,
    warmup_epsilon: float,
    grad_steps_per_epoch: int = 4,
    agent_label: str = "agent",
    verbose: bool = True,
) -> pd.DataFrame:
    """Train agent on each scenario combo, recording per-epoch metrics.

    For trainable agents: runs warmup_epochs episodes per combo with
    exploration and gradient steps, recording metrics each epoch.
    Training is cumulative across combos (weights not restored).

    For non-trainable agents (heuristic): runs 1 episode per combo.
    """
    runner = TutorialRunner(
        env_factory=env_factory,
        agent_or_config=agent,
        verbose=False,
    )

    is_trainable = getattr(agent, 'is_trainable', True)

    if is_trainable and hasattr(agent, 'epsilon_override'):
        agent.epsilon_override = warmup_epsilon
    if hasattr(agent, 'set_noise_scale'):
        agent.set_noise_scale(1.0)

    rows = []
    n_epochs = warmup_epochs if is_trainable else 1
    total = len(param_grid) * n_epochs
    run_num = 0

    for p_idx, params in enumerate(param_grid):
        for epoch in range(n_epochs):
            ep_params = EvalParams(
                n_imports=params.n_imports,
                n_exports=params.n_exports,
                n_delivery_trucks=params.n_delivery_trucks,
                n_pickup_trucks=params.n_pickup_trucks,
                n_train_to_truck=params.n_train_to_truck,
                n_truck_to_train=params.n_truck_to_train,
                yard_fill_pct=params.yard_fill_pct,
                seed=params.seed + epoch * 137,
                safety_factor=params.safety_factor,
            )
            scenario = ScalableEvalScenario(ep_params)

            t0 = time.time()
            result = runner.run_scenario(scenario)
            wall_time = time.time() - t0

            progress = scenario.check_progress(runner.env)

            # Gradient steps (only for trainable agents with enough data)
            if is_trainable and hasattr(agent, 'optimize'):
                if hasattr(agent, 'replay') and hasattr(agent.replay, 'is_ready'):
                    if agent.replay.is_ready(agent.cfg.training.batch_size):
                        for _ in range(grad_steps_per_epoch):
                            agent.optimize()

            # Compute completion
            total_done = sum(st.completed for st in progress)
            total_needed = sum(st.total for st in progress)
            completion_pct = total_done / max(1, total_needed)

            # Move counts
            mc = result.move_type_counts
            total_moves = result.agent_moves
            reshuffle_count = mc.get('YARD_TO_YARD', 0)

            row = {
                'agent': agent_label,
                'epoch': epoch,
                'n_imports': params.n_imports,
                'n_exports': params.n_exports,
                'n_delivery_trucks': params.n_delivery_trucks,
                'n_pickup_trucks': params.n_pickup_trucks,
                'n_train_to_truck': params.n_train_to_truck,
                'n_truck_to_train': params.n_truck_to_train,
                'total_containers': params.total_containers,
                'yard_fill_pct': params.yard_fill_pct,
                'seed': ep_params.seed,
                'passed': result.passed,
                'completion_pct': completion_pct,
                'steps': result.steps,
                'agent_moves': total_moves,
                'total_reward': result.total_reward,
                'moves_per_container': total_moves / max(1, params.total_containers),
                'reshuffle_ratio': reshuffle_count / max(1, total_moves),
                'wall_time_s': wall_time,
            }

            for st in progress:
                row[f'{st.name}_pct'] = st.pct
                row[f'{st.name}_completed'] = st.completed
                row[f'{st.name}_total'] = st.total

            for mt, count in mc.items():
                row[f'moves_{mt}'] = count

            rows.append(row)
            run_num += 1

            if verbose and (run_num % 5 == 0 or run_num == total):
                tag = 'PASS' if result.passed else 'FAIL'
                pcts = ' '.join(f'{st.name}={st.pct:.0%}' for st in progress)
                print(
                    f'  [{run_num:3d}/{total}] {tag} '
                    f'ep={epoch:2d} {params.label} '
                    f'compl={completion_pct:.0%} '
                    f'R={result.total_reward:+.1f} [{pcts}]'
                )

    if is_trainable and hasattr(agent, 'clear_epsilon_override'):
        agent.clear_epsilon_override()

    return pd.DataFrame(rows)


# ── Run warmup curves for all DQN agents ──────────────────────
all_curves: Dict[str, pd.DataFrame] = {}

for label, agent in agents.items():
    print(f'\n{"="*60}')
    print(f'Warmup curves: {label}')
    print(f'{"="*60}')

    t0 = time.time()
    df = run_warmup_curves(
        agent, env_factory, eval_grid,
        warmup_epochs=WARMUP_EPOCHS,
        warmup_epsilon=WARMUP_EPSILON,
        grad_steps_per_epoch=WARMUP_GRAD_STEPS,
        agent_label=label,
        verbose=True,
    )
    elapsed = time.time() - t0
    print(f'  {len(df)} rows in {elapsed:.1f}s')
    all_curves[label] = df

# ── Heuristic baseline (single run per combo) ────────────────
print(f'\n{"="*60}')
print(f'Heuristic baseline')
print(f'{"="*60}')

t0 = time.time()
heuristic_df = run_warmup_curves(
    heuristic, env_factory, eval_grid,
    warmup_epochs=1,
    warmup_epsilon=0.0,
    agent_label='heuristic',
    verbose=True,
)
elapsed = time.time() - t0
print(f'  {len(heuristic_df)} rows in {elapsed:.1f}s')
all_curves['heuristic'] = heuristic_df

print(f'\nCollected curves for {len(all_curves)} agents.')


Warmup curves: baseline_baseline


/home/franko/.cache/pypoetry/virtualenvs/ct-drl-ma-7RJlkqYG-py3.12/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator KernelDensity from version 1.5.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  [  5/450] PASS ep= 4 imp=10_exp=10_fill=0% compl=100% R=+150.4 [imports_unloaded=100% exports_loaded=100%]
  [ 10/450] FAIL ep= 9 imp=10_exp=10_fill=0% compl=90% R=+52.0 [imports_unloaded=100% exports_loaded=80%]
  [ 15/450] PASS ep=14 imp=10_exp=10_fill=0% compl=100% R=+151.0 [imports_unloaded=100% exports_loaded=100%]


KeyboardInterrupt: 

In [ ]:
# ================================================================
# Cell 7: Summary Statistics
# ================================================================

for label, df in all_curves.items():
    print(f'\n{"="*60}')
    print(f'Summary: {label}')
    print(f'{"="*60}')

    is_heuristic = (label == 'heuristic')

    if not is_heuristic:
        first = df[df['epoch'] == 0]
        last = df[df['epoch'] == df['epoch'].max()]
        print(f'  First epoch:  completion={first["completion_pct"].mean():.1%}  '
              f'pass={first["passed"].mean():.0%}  '
              f'reward={first["total_reward"].mean():+.1f}')
        print(f'  Last epoch:   completion={last["completion_pct"].mean():.1%}  '
              f'pass={last["passed"].mean():.0%}  '
              f'reward={last["total_reward"].mean():+.1f}')
        print(f'  Improvement:  '
              f'{last["completion_pct"].mean() - first["completion_pct"].mean():+.1%}')
    else:
        print(f'  Completion: {df["completion_pct"].mean():.1%}  '
              f'pass={df["passed"].mean():.0%}  '
              f'reward={df["total_reward"].mean():+.1f}')

    # Per-combo breakdown (final epoch for DQN, single run for heuristic)
    src = last if not is_heuristic else df
    group_cols = ['total_containers', 'yard_fill_pct']
    grp = src.groupby(group_cols).agg(
        completion=('completion_pct', 'mean'),
        pass_rate=('passed', 'mean'),
        reward=('total_reward', 'mean'),
        moves_per_c=('moves_per_container', 'mean'),
        reshuffle=('reshuffle_ratio', 'mean'),
    ).reset_index()

    tag = 'final epoch' if not is_heuristic else 'single run'
    print(f'\n  Per-combo ({tag}):')
    print(grp.to_string(index=False, float_format='%.3f'))

    # Sub-task averages
    sub_task_cols = [c for c in src.columns
                     if c.endswith('_pct') and c != 'completion_pct']
    if sub_task_cols:
        print(f'\n  Sub-task averages ({tag}):')
        for col in sorted(sub_task_cols):
            vals = src[col].dropna()
            if len(vals) > 0:
                print(f'    {col:35s} {vals.mean():.1%}')

In [ ]:
# ================================================================
# Cell 8: Thesis Figures
# ================================================================

# ── Colour & styling ──────────────────────────────────────────
_COLOR_CYCLE = ['#2196F3', '#E91E63', '#4CAF50', '#9C27B0', '#00BCD4', '#795548']
AGENT_COLORS = {}
for i, (v, b) in enumerate(AGENTS_TO_EVAL):
    AGENT_COLORS[agent_label(v, b)] = _COLOR_CYCLE[i % len(_COLOR_CYCLE)]
AGENT_COLORS['heuristic'] = '#FF9800'

FILL_STYLES = {0.0: '-', 0.25: '--', 0.50: ':'}
FILL_LABELS = {0.0: '0%', 0.25: '25%', 0.50: '50%'}

SUBTASK_COLORS = {
    'imports_unloaded_pct': '#2196F3',
    'exports_loaded_pct': '#4CAF50',
    'deliveries_served_pct': '#FF9800',
    'pickups_served_pct': '#E91E63',
    'train_to_truck_done_pct': '#9C27B0',
    'truck_to_train_done_pct': '#00BCD4',
}
SUBTASK_LABELS = {
    'imports_unloaded_pct': 'Imports Unloaded',
    'exports_loaded_pct': 'Exports Loaded',
    'deliveries_served_pct': 'Deliveries Served',
    'pickups_served_pct': 'Pickups Served',
    'train_to_truck_done_pct': 'Train\u2192Truck',
    'truck_to_train_done_pct': 'Truck\u2192Train',
}

def _color(label):
    return AGENT_COLORS.get(label, '#607D8B')


# ────────────────────────────────────────────────────────────────
# Figure 1: Completion Learning Curves (faceted by load size)
# ────────────────────────────────────────────────────────────────

def plot_learning_curves(all_curves, smooth_window=3):
    """One subplot per load size. Completion % vs. warmup epoch."""
    # Collect all unique load sizes across all agents
    all_loads = set()
    for df in all_curves.values():
        all_loads.update(int(v) for v in df['total_containers'].unique())
    load_sizes = sorted(all_loads)

    n = len(load_sizes)
    n_cols = min(3, n)
    n_rows = (n + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows),
                             squeeze=False)

    dqn_labels = [l for l in all_curves if l != 'heuristic']

    for ax_idx, load in enumerate(load_sizes):
        ax = axes[ax_idx // n_cols, ax_idx % n_cols]

        # DQN agent curves
        for label in dqn_labels:
            df = all_curves[label]
            color = _color(label)
            sub = df[df['total_containers'] == load]

            for fill in sorted(sub['yard_fill_pct'].unique()):
                fill_sub = sub[sub['yard_fill_pct'] == fill].sort_values('epoch')
                if len(fill_sub) == 0:
                    continue
                fill_lbl = FILL_LABELS.get(fill, f'{fill:.0%}')
                style = FILL_STYLES.get(fill, '-.')

                y = fill_sub['completion_pct'].values
                x = fill_sub['epoch'].values
                if len(y) >= smooth_window:
                    y_smooth = pd.Series(y).rolling(
                        smooth_window, min_periods=1).mean().values
                else:
                    y_smooth = y

                lbl = f'{label} fill={fill_lbl}' if len(dqn_labels) > 1 else f'fill={fill_lbl}'
                ax.plot(x, y_smooth, linestyle=style, color=color,
                        linewidth=1.5, label=lbl)
                ax.scatter(x, y, color=color, alpha=0.12, s=6, zorder=1)

        # Heuristic horizontal lines
        if 'heuristic' in all_curves:
            h_df = all_curves['heuristic']
            h_sub = h_df[h_df['total_containers'] == load]
            for fill in sorted(h_sub['yard_fill_pct'].unique()):
                h_val = h_sub[h_sub['yard_fill_pct'] == fill]['completion_pct'].mean()
                fill_lbl = FILL_LABELS.get(fill, f'{fill:.0%}')
                style = FILL_STYLES.get(fill, '-.')
                ax.axhline(h_val, color=_color('heuristic'),
                          linestyle=style, alpha=0.6, linewidth=1.2,
                          label=f'heuristic fill={fill_lbl}')

        ax.set_title(f'{load} containers')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Completion %')
        ax.set_ylim(-0.05, 1.05)
        ax.grid(True, alpha=0.3)
        if ax_idx == 0:
            ax.legend(fontsize=6, loc='lower right')

    for idx in range(n, n_rows * n_cols):
        axes[idx // n_cols, idx % n_cols].set_visible(False)

    fig.suptitle('Warmup Learning Curves: Completion % vs. Epoch',
                 fontsize=13, y=1.01)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'learning_curves.png', bbox_inches='tight')
    plt.show()

plot_learning_curves(all_curves)


# ────────────────────────────────────────────────────────────────
# Figure 2: Sub-Task Learning Curves (easy vs hard scenarios)
# ────────────────────────────────────────────────────────────────

def plot_subtask_curves(all_curves, smooth_window=3):
    """Per-sub-task completion over epochs for easy and hard scenarios."""
    sizes = sorted(LOAD_SIZES)
    scenarios = [
        (sizes[0], 0.0, 'Easy'),
        (sizes[-1], 0.50, 'Hard'),
    ]
    dqn_labels = [l for l in all_curves if l != 'heuristic']
    if not dqn_labels:
        return

    ref_df = all_curves[dqn_labels[0]]
    avail_st = [c for c in ref_df.columns
                if c.endswith('_pct') and c != 'completion_pct']

    n_agents = len(dqn_labels)
    n_scenarios = len(scenarios)
    fig, axes = plt.subplots(n_scenarios, max(1, n_agents),
                             figsize=(6 * n_agents, 4 * n_scenarios),
                             squeeze=False)

    for s_idx, (load, fill, tag) in enumerate(scenarios):
        for a_idx, label in enumerate(dqn_labels):
            ax = axes[s_idx, a_idx]
            df = all_curves[label]
            sub = df[(df['total_containers'] == load)
                     & (df['yard_fill_pct'] == fill)].sort_values('epoch')

            if len(sub) == 0:
                ax.set_visible(False)
                continue

            for st_col in avail_st:
                if st_col not in sub.columns:
                    continue
                vals = sub[st_col].dropna()
                if len(vals) == 0:
                    continue

                y = vals.values
                x = sub['epoch'].values[:len(y)]
                if len(y) >= smooth_window:
                    y_smooth = pd.Series(y).rolling(
                        smooth_window, min_periods=1).mean().values
                else:
                    y_smooth = y

                color = SUBTASK_COLORS.get(st_col, '#999')
                st_label = SUBTASK_LABELS.get(st_col, st_col.replace('_pct', ''))
                ax.plot(x, y_smooth, color=color, linewidth=1.5, label=st_label)
                ax.scatter(sub['epoch'].values[:len(y)], y,
                          color=color, alpha=0.12, s=6, zorder=1)

            # Heuristic reference lines per sub-task
            if 'heuristic' in all_curves:
                h_df = all_curves['heuristic']
                h_sub = h_df[(h_df['total_containers'] == load)
                             & (h_df['yard_fill_pct'] == fill)]
                for st_col in avail_st:
                    if st_col in h_sub.columns and len(h_sub[st_col].dropna()) > 0:
                        h_val = h_sub[st_col].mean()
                        color = SUBTASK_COLORS.get(st_col, '#999')
                        ax.axhline(h_val, color=color, linestyle=':', alpha=0.5)

            fill_lbl = FILL_LABELS.get(fill, f'{fill:.0%}')
            title = f'{tag}: {load} containers, fill={fill_lbl}'
            if n_agents > 1:
                title = f'{label}\n{title}'
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Sub-Task %')
            ax.set_ylim(-0.05, 1.05)
            ax.grid(True, alpha=0.3)
            if s_idx == 0 and a_idx == 0:
                ax.legend(fontsize=7, loc='lower right')

    fig.suptitle('Sub-Task Completion Trends', fontsize=13, y=1.01)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'subtask_curves.png', bbox_inches='tight')
    plt.show()

plot_subtask_curves(all_curves)


# ────────────────────────────────────────────────────────────────
# Figure 3: Scaling Curve (final epoch completion vs scenario size)
# ────────────────────────────────────────────────────────────────

def plot_scaling_curve(all_curves):
    """Final-epoch completion vs. total containers for all agents."""
    fig, ax = plt.subplots(figsize=(8, 5))

    for label, df in all_curves.items():
        color = _color(label)
        is_heuristic = (label == 'heuristic')

        if is_heuristic:
            data = df
        else:
            data = df[df['epoch'] == df['epoch'].max()]

        for fill in sorted(data['yard_fill_pct'].unique()):
            sub = data[data['yard_fill_pct'] == fill]
            grp = sub.groupby('total_containers')['completion_pct'].mean().sort_index()

            fill_lbl = FILL_LABELS.get(fill, f'{fill:.0%}')
            style = FILL_STYLES.get(fill, '-.')
            marker = 's' if is_heuristic else 'o'

            ax.plot(grp.index, grp.values,
                    marker=marker, markersize=5,
                    linestyle=style, color=color, linewidth=1.5,
                    label=f'{label} fill={fill_lbl}')

    ax.set_xlabel('Total Containers')
    ax.set_ylabel('Completion %')
    ax.set_title('Completion Rate vs. Scenario Size (Final Epoch)')
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(1.0, color='gray', linestyle=':', alpha=0.3)
    ax.legend(fontsize=7, loc='lower left')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'scaling_curve.png', bbox_inches='tight')
    plt.show()

plot_scaling_curve(all_curves)


# ────────────────────────────────────────────────────────────────
# Figure 4: Completion Heatmap (final epoch)
# ────────────────────────────────────────────────────────────────

def plot_completion_heatmap(all_curves):
    """Heatmap of completion % by load x fill, one per agent."""
    agent_labels = list(all_curves.keys())
    n_agents = len(agent_labels)

    fig, axes = plt.subplots(1, n_agents, figsize=(4.5 * n_agents, 3.5),
                             squeeze=False)

    for idx, label in enumerate(agent_labels):
        ax = axes[0, idx]
        df = all_curves[label]

        if label == 'heuristic':
            data = df
        else:
            data = df[df['epoch'] == df['epoch'].max()]

        fills = sorted(data['yard_fill_pct'].unique())
        loads = sorted(data['total_containers'].unique())

        matrix = np.full((len(fills), len(loads)), np.nan)
        for i, fill in enumerate(fills):
            for j, load in enumerate(loads):
                sub = data[(data['yard_fill_pct'] == fill)
                           & (data['total_containers'] == load)]
                if len(sub) > 0:
                    matrix[i, j] = sub['completion_pct'].mean()

        im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1,
                       aspect='auto', origin='lower')
        ax.set_xticks(range(len(loads)))
        ax.set_xticklabels([str(int(l)) for l in loads], fontsize=8)
        ax.set_yticks(range(len(fills)))
        ax.set_yticklabels([FILL_LABELS.get(f, f'{f:.0%}') for f in fills])
        ax.set_xlabel('Total Containers')
        ax.set_ylabel('Yard Fill')

        title = label.replace('_', ' ').title()
        if label != 'heuristic':
            title += f' (ep {int(df["epoch"].max())})'
        ax.set_title(title)

        for i in range(len(fills)):
            for j in range(len(loads)):
                val = matrix[i, j]
                if not np.isnan(val):
                    color = 'white' if val < 0.5 else 'black'
                    ax.text(j, i, f'{val:.0%}', ha='center', va='center',
                            fontsize=9, fontweight='bold', color=color)

    fig.colorbar(im, ax=axes.ravel().tolist(), label='Completion %',
                 shrink=0.8, pad=0.02)
    fig.suptitle('Completion Rate Heatmap', fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'completion_heatmap.png', bbox_inches='tight')
    plt.show()

plot_completion_heatmap(all_curves)


# ────────────────────────────────────────────────────────────────
# Figure 5: Reward Learning Curves
# ────────────────────────────────────────────────────────────────

def plot_reward_curves(all_curves, smooth_window=5):
    """Average reward across all scenarios vs. epoch."""
    dqn_labels = [l for l in all_curves if l != 'heuristic']
    if not dqn_labels:
        return

    fig, ax = plt.subplots(figsize=(8, 5))

    for label in dqn_labels:
        df = all_curves[label]
        color = _color(label)
        grp = df.groupby('epoch')['total_reward'].mean()
        y = grp.values
        if len(y) >= smooth_window:
            y_smooth = pd.Series(y).rolling(
                smooth_window, min_periods=1).mean().values
        else:
            y_smooth = y
        ax.plot(grp.index, y_smooth, color=color, linewidth=1.5, label=label)
        ax.scatter(grp.index, y, color=color, alpha=0.15, s=10, zorder=1)

    # Heuristic average
    if 'heuristic' in all_curves:
        h_mean = all_curves['heuristic']['total_reward'].mean()
        ax.axhline(h_mean, color=_color('heuristic'), linestyle='--',
                   alpha=0.7, label=f'heuristic (avg={h_mean:+.1f})')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Average Reward')
    ax.set_title('Reward Learning Curve (averaged across scenarios)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'reward_curves.png', bbox_inches='tight')
    plt.show()

plot_reward_curves(all_curves)


# ────────────────────────────────────────────────────────────────
# Figure 6: Move Distribution (final epoch)
# ────────────────────────────────────────────────────────────────

def plot_move_distribution(all_curves):
    """Stacked bar chart of move types at fill=0%, final epoch."""
    move_types = [
        'YARD_TO_TRAIN', 'TRAIN_TO_YARD', 'PARK_TRUCK',
        'TRUCK_TO_YARD', 'YARD_TO_TRUCK', 'YARD_TO_YARD',
        'TRAIN_TO_TRUCK', 'TRUCK_TO_TRAIN',
    ]
    move_colors = {
        'YARD_TO_TRAIN': '#4CAF50',
        'TRAIN_TO_YARD': '#2196F3',
        'PARK_TRUCK': '#FF9800',
        'TRUCK_TO_YARD': '#9C27B0',
        'YARD_TO_TRUCK': '#E91E63',
        'YARD_TO_YARD': '#F44336',
        'TRAIN_TO_TRUCK': '#00BCD4',
        'TRUCK_TO_TRAIN': '#795548',
    }

    agent_labels = list(all_curves.keys())
    n_agents = len(agent_labels)
    fig, axes = plt.subplots(1, n_agents, figsize=(5 * n_agents, 4.5),
                             squeeze=False, sharey=True)

    for idx, label in enumerate(agent_labels):
        ax = axes[0, idx]
        df = all_curves[label]

        if label == 'heuristic':
            data = df[df['yard_fill_pct'] == 0.0]
        else:
            max_ep = df['epoch'].max()
            data = df[(df['epoch'] == max_ep) & (df['yard_fill_pct'] == 0.0)]

        if len(data) == 0:
            data = df if label == 'heuristic' else df[df['epoch'] == max_ep]

        loads = sorted(data['total_containers'].unique())
        x = np.arange(len(loads))
        width = 0.6

        bottoms = np.zeros(len(loads))
        for mt in move_types:
            col = f'moves_{mt}'
            if col not in data.columns:
                continue
            means = [data[data['total_containers'] == l][col].mean()
                     for l in loads]
            means = [m if not np.isnan(m) else 0 for m in means]
            ax.bar(x, means, width, bottom=bottoms,
                   label=mt.replace('_', ' ').title(),
                   color=move_colors.get(mt, '#999'))
            bottoms += np.array(means)

        ax.set_xticks(x)
        ax.set_xticklabels([str(int(l)) for l in loads])
        ax.set_xlabel('Total Containers')
        ax.set_ylabel('Avg. Move Count')
        title = label.replace('_', ' ').title()
        if label != 'heuristic':
            title += ' (fill=0%)'
        ax.set_title(title)

    axes[0, 0].legend(fontsize=7, loc='upper left')
    fig.suptitle('Move Type Distribution by Scenario Size', fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'move_distribution.png', bbox_inches='tight')
    plt.show()

plot_move_distribution(all_curves)

In [ ]:
# ================================================================
# Cell 9: Export Results
# ================================================================

# Save per-agent learning curve data
for label, df in all_curves.items():
    path = OUTPUT_DIR / f'warmup_curves_{label}.csv'
    df.to_csv(path, index=False)
    print(f'Saved: {path}')

# Combined comparison table (final epoch for DQN, single run for heuristic)
comparison_rows = []
for label, df in all_curves.items():
    if label == 'heuristic':
        data = df
    else:
        data = df[df['epoch'] == df['epoch'].max()]

    comparison_rows.append({
        'agent': label,
        'total_runs': len(data),
        'pass_rate': data['passed'].mean(),
        'avg_completion': data['completion_pct'].mean(),
        'avg_moves_per_container': data['moves_per_container'].mean(),
        'avg_reshuffle_ratio': data['reshuffle_ratio'].mean(),
        'avg_reward': data['total_reward'].mean(),
    })

comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(OUTPUT_DIR / 'agent_comparison.csv', index=False)
print(f'\nSaved comparison: {OUTPUT_DIR / "agent_comparison.csv"}')
print('\n--- Agent Comparison (Final Epoch vs. Heuristic) ---')
print(comparison.to_string(index=False, float_format='%.3f'))

print(f'\nAll figures saved to: {FIG_DIR}')
print('Done!')